In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

df = pd.read_csv("../data/carclaims.csv")
print("Shape:", df.shape)
df.head(2)


Shape: (15420, 33)


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange-Claim,NumberOfCars,Year,BasePolicy,FraudFound
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability,No
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision,No


In [7]:
feature_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "unique_values": [
        df[column].nunique()
        for column in df.columns
    ]
})

display(feature_summary)

,column,dtype,unique_values
0,Month,str,12
1,WeekOfMonth,int64,5
2,DayOfWeek,str,7
3,Make,str,19
4,AccidentArea,str,2
5,DayOfWeekClaimed,str,8
6,MonthClaimed,str,13
7,WeekOfMonthClaimed,int64,5
8,Sex,str,2
9,MaritalStatus,str,4


In [8]:
# For now, we'll exclude:
# - FraudFound from the input features because it is our target.
# - PolicyNumber because it appears to be an identifier.
# - Fault because we have not yet established whether it is available at the intended prediction time.
# - RepNumber because it may encode representative-specific patterns that need separate investigation.


In [12]:
# PolicyNumber = just an ID, RepNumber = agent ID, Year = almost no variation
columns_to_drop = ["PolicyNumber", "RepNumber", "Year", "FraudFound"]

y = df["FraudFound"].map({"Yes": 1, "No": 0})
X = df.drop(columns=columns_to_drop)

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).mul(100).round(2).astype(str) + "%")


Features shape: (15420, 29)
Target distribution:
FraudFound
0    94.01%
1     5.99%
Name: proportion, dtype: str


Train-Test-Split

In [14]:


X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training features: (12336, 29)
Testing features: (3084, 29)

Training target distribution:
FraudFound
0    94.02
1     5.98
Name: proportion, dtype: float64

Testing target distribution:
FraudFound
0    94.0
1     6.0
Name: proportion, dtype: float64


In [15]:
numerical_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:", len(numerical_columns))
print(numerical_columns)

print("\nCategorical columns:", len(categorical_columns))
print(categorical_columns)


Numerical columns: 5
['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'Deductible', 'DriverRating']

Categorical columns: 24
['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days:Policy-Accident', 'Days:Policy-Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange-Claim', 'NumberOfCars', 'BasePolicy']


Building Preprocessing Pipeline

In [16]:
numerical_transformer=Pipeline(
    steps=[
        ("scaler",StandardScaler())
    ]
)
categorical_transformer=Pipeline(
    steps=[
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)
# preprocessing done passing them to transformer 
transformers=[
    ("num",numerical_transformer,numerical_columns),
    ("cat",categorical_transformer, categorical_columns)
]
preprocessor=ColumnTransformer(
    transformers=transformers,
    remainder="passthrough"
    )
print("Preprocessor created successfully!")

Preprocessor created successfully!
